# Batch Scoring — Dropout Risk Prediction

Scores every enrolment; writes predictions + per-department dropout risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/dropout_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'credits', 'age_at_enrolment', 'avg_score', 'pass_rate',
    'assessment_count', 'cohort_year',
]
cat_cols = ['department', 'level', 'programme', 'gender', 'region']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('dropout_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_dropout', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('dropout_probability') > 0.8, 'critical')
        .when(col('dropout_probability') > 0.6, 'high')
        .when(col('dropout_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'enrolment_id', 'student_id', 'department', 'level', 'programme',
        'credits', 'age_at_enrolment', 'avg_score', 'pass_rate',
        'had_dropout', 'predicted_dropout', 'dropout_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-department dropout risk summary
summary = (
    predictions
    .groupBy('department')
    .agg(
        count('*').alias('total_enrolments'),
        spark_sum('predicted_dropout').alias('predicted_dropout_count'),
        spark_sum('had_dropout').alias('actual_dropout_count'),
        spark_round(avg('dropout_probability'), 4).alias('avg_dropout_probability'),
        spark_round(avg('avg_score'), 1).alias('avg_assessment_score'),
        spark_round(avg('age_at_enrolment'), 1).alias('avg_age'),
    )
    .withColumn('dropout_rate', spark_round(col('predicted_dropout_count') / col('total_enrolments') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_dropout_probability') > 0.6, 'high')
        .when(col('avg_dropout_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_dropout_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Department dropout summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')